In [1]:
import os
import time
import torch
import torch.nn as nn
import torch.optim
from torch.utils.data import DataLoader 
from chess import pgn
import chess
from tqdm import tqdm
from pathlib import  Path
from data_extraction import  board_to_tensor




In [3]:

pgn_dir = Path(r"D:\Coding\CNN Chess Engine\data\gmdata")
unique_moves = set()

X = []
Y= []

def load_pgn(file_path):
    games =[]
    with open(file_path,'r') as pgn_file:
        while True:
            game = chess.pgn.read_game(pgn_file)
            if game is None:
                break
            for move in game.mainline_moves():
                unique_moves.add(move.uci())
            
            games.append(game)

    return games


files=[]
for file in pgn_dir.glob("*.pgn"):
    files.append(file)


games=[]
for file in tqdm(files[:30]):
    games.extend(load_pgn(file))

move_to_idx = {
    move : idx for idx,move in enumerate(sorted(unique_moves))
}

idx_to_move = {
    idx : move for idx,move in enumerate(sorted(unique_moves))
}

torch.save(
    move_to_idx,
    r"D:\Coding\CNN Chess Engine\data\gmdata\move_to_idx.pt"
)

torch.save(
    idx_to_move,
    r"D:\Coding\CNN Chess Engine\data\gmdata\idx_to_move.pt"
)

100%|██████████| 30/30 [03:52<00:00,  7.75s/it]


In [6]:
X = []
Y = []

CHUNK_SIZE = 500_000
MAX_POSITIONS = 15_000_000

chunk_id = 1
position_count = 0

for game in tqdm(games):

    board = game.board()

    for move in game.mainline_moves():

        X.append(board_to_tensor(board).to(torch.uint8))
        Y.append(move_to_idx[move.uci()])

        board.push(move)

        position_count += 1

        if len(X) >= CHUNK_SIZE:

            print(f"Saved : {chunk_id}")

            X_tensor = torch.stack(X)
            Y_tensor = torch.tensor(Y, dtype=torch.long)

            torch.save(
                X_tensor,
                fr"D:\Coding\CNN Chess Engine\data\gmdata\X_{chunk_id}.pt"
            )

            torch.save(
                Y_tensor,
                fr"D:\Coding\CNN Chess Engine\data\gmdata\Y_{chunk_id}.pt"
            )

            X.clear()
            Y.clear()

            chunk_id += 1

        if position_count >= MAX_POSITIONS:
            break

    if position_count >= MAX_POSITIONS:
        break

  7%|▋         | 5927/81313 [03:25<42:17, 29.71it/s]  

Saved : 1


 15%|█▍        | 12186/81313 [06:54<38:56, 29.58it/s]  

Saved : 2


 23%|██▎       | 18375/81313 [10:26<32:04, 32.70it/s]  

...

 85%|████████▍ | 68876/81313 [39:42<06:15, 33.12it/s]  

Saved : 11


 92%|█████████▏| 75051/81313 [42:55<03:17, 31.65it/s]  

Saved : 12


100%|██████████| 81313/81313 [46:06<00:00, 29.39it/s]
